In [ ]:
# CÉLULA 0 — AWS / SAGEMAKER: dependências do notebook
# %pip instala no mesmo kernel Jupyter em uso e pode ser executado com segurança novamente.
%pip install -q --disable-pip-version-check yfinance pyarrow pandas numpy

print("Dependências instaladas/verificadas. Continue para a próxima célula.")


# 20.22A — Yahoo Finance: dataset multiativo unificado, temporal e pronto para Conformal Prediction

**Versão v2 AWS:** corrige a elegibilidade para calendários heterogêneos, elimina `ffill` global e audita `N_joint` por Hamiltoniano.\n\n## Objetivo

Este notebook cria um **dataset reprodutível e com nomes canônicos já definidos** para a expansão do experimento 20.21/20.22.

Estratégia padrão:

- baixar **um ano completo** do Yahoo Finance para desenvolvimento;
- separar internamente esse ano em **treino + calibração conformal**;
- usar **o ano seguinte inteiro como teste temporal fora da amostra**;
- incluir ativos de diferentes classes;
- impedir vazamento temporal;
- criar `asset_id`, `parent_instance_id`, `scenario_uid` e nomes de arquivos estáveis;
- preparar subconjuntos com diferentes `n` e `k/n`;
- deixar a estrutura pronta para VQE/QGT/cold-start/Transformer/Conformal Prediction.

### Split temporal padrão

\[
\text{ano }Y:
\quad
\underbrace{\text{jan--set}}_{\text{train}}
+
\underbrace{\text{out--dez}}_{\text{calibration}}
\]

\[
\text{ano }Y+1:
\quad
\underbrace{\text{jan--dez}}_{\text{test}}
\]

O **ano de teste nunca participa** da seleção de ativos, calibração ou normalização.

## Regra de nomenclatura

Cada ativo possui um identificador canônico independente do ticker do Yahoo:

```text
BR_EQ_PETR4
US_EQ_AAPL
ETF_US_SPY
IDX_BR_IBOV
FX_USDBRL
CRYPTO_BTCUSD
CMDTY_GOLD
```

Arquivos seguem:

```text
yf__<tipo>__<split>__v1.<ext>
```

Exemplos:

```text
yf__prices__train__v1.parquet
yf__returns__calibration__v1.parquet
yf__returns__test__v1.parquet
asset_registry__v1.csv
parent_instances__v1.csv
dataset_manifest__v1.json
```


In [ ]:
# CÉLULA 1 — imports e ambiente
from pathlib import Path
from hashlib import sha256
from datetime import datetime
import json
import time
import warnings

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("yfinance:", yf.__version__)


## Célula 2 — configuração central

Altere apenas esta célula para mudar os anos, a escala do experimento ou o universo de ativos.

**Padrão sugerido:** treinar/calibrar em 2024 e testar em 2025.

Para um teste futuro, basta trocar para:

```python
TRAIN_YEAR = 2025
TEST_YEAR  = 2026
```

Quando o ano de teste ainda estiver em andamento, o Yahoo entregará apenas os dados disponíveis até a data corrente.


In [ ]:
# CÉLULA 2 — CONFIGURAÇÃO CENTRAL

DATASET_VERSION = "v2"
PROVIDER = "yahoo_finance"

TRAIN_YEAR = 2024
TEST_YEAR  = 2025
CALIBRATION_START_MONTH = 10

N_VALUES = [10, 12, 14, 16, 18, 20]
K_OVER_N_VALUES = [0.20, 0.30, 0.40, 0.50, 0.60]
N_PARENT_PER_COMBO = 20

RANDOM_SEED = 20260828

# Elegibilidade por calendário natural do ativo.
# Não usar cobertura sobre o índice global misto (ações + cripto + FX etc.).
MIN_DEV_OBSERVATIONS = 200
MIN_TRAIN_OBSERVATIONS = 150
MIN_CALIBRATION_OBSERVATIONS = 40
BOUNDARY_TOLERANCE_DAYS = 15

# Número mínimo de retornos simultaneamente observados para estimar mu/Sigma
# em um parent instance.
MIN_JOINT_TRAIN_OBSERVATIONS = 120

# "MIXED_GLOBAL", "EQUITY_ONLY", "BRAZIL_ONLY", "US_LIQUID", "MULTI_ASSET"
UNIVERSE_MODE = "MIXED_GLOBAL"
SAVE_CSV_TOO = True

OUTPUT_ROOT = Path("20_22A_yahoo_dataset_v2")
DIRS = {
    "registry": OUTPUT_ROOT / "00_registry",
    "raw": OUTPUT_ROOT / "01_raw",
    "processed": OUTPUT_ROOT / "02_processed",
    "splits": OUTPUT_ROOT / "03_splits",
    "instances": OUTPUT_ROOT / "04_instances",
    "manifests": OUTPUT_ROOT / "05_manifests",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

print("OUTPUT_ROOT =", OUTPUT_ROOT.resolve())


## Célula 3 — registro canônico dos ativos

O `asset_id` é o **nome definitivo usado no projeto**. O `yahoo_ticker` existe apenas para comunicação com o provedor.


In [ ]:
# CÉLULA 3 — ASSET REGISTRY CANÔNICO

ASSETS = [
    # Brasil — ações
    ("BR_EQ_PETR4", "PETR4.SA", "equity", "BR", "BRL", "Petrobras PN"),
    ("BR_EQ_VALE3", "VALE3.SA", "equity", "BR", "BRL", "Vale ON"),
    ("BR_EQ_ITUB4", "ITUB4.SA", "equity", "BR", "BRL", "Itau Unibanco PN"),
    ("BR_EQ_BBDC4", "BBDC4.SA", "equity", "BR", "BRL", "Bradesco PN"),
    ("BR_EQ_BBAS3", "BBAS3.SA", "equity", "BR", "BRL", "Banco do Brasil ON"),
    ("BR_EQ_ABEV3", "ABEV3.SA", "equity", "BR", "BRL", "Ambev ON"),
    ("BR_EQ_WEGE3", "WEGE3.SA", "equity", "BR", "BRL", "WEG ON"),
    ("BR_EQ_B3SA3", "B3SA3.SA", "equity", "BR", "BRL", "B3 ON"),
    ("BR_EQ_SUZB3", "SUZB3.SA", "equity", "BR", "BRL", "Suzano ON"),
    ("BR_EQ_RENT3", "RENT3.SA", "equity", "BR", "BRL", "Localiza ON"),
    ("BR_EQ_PRIO3", "PRIO3.SA", "equity", "BR", "BRL", "PRIO ON"),
    ("BR_EQ_ELET3", "ELET3.SA", "equity", "BR", "BRL", "Eletrobras ON"),

    # EUA — ações
    ("US_EQ_AAPL", "AAPL", "equity", "US", "USD", "Apple"),
    ("US_EQ_MSFT", "MSFT", "equity", "US", "USD", "Microsoft"),
    ("US_EQ_NVDA", "NVDA", "equity", "US", "USD", "NVIDIA"),
    ("US_EQ_AMZN", "AMZN", "equity", "US", "USD", "Amazon"),
    ("US_EQ_GOOGL", "GOOGL", "equity", "US", "USD", "Alphabet"),
    ("US_EQ_META", "META", "equity", "US", "USD", "Meta"),
    ("US_EQ_JPM", "JPM", "equity", "US", "USD", "JPMorgan"),
    ("US_EQ_XOM", "XOM", "equity", "US", "USD", "Exxon Mobil"),
    ("US_EQ_JNJ", "JNJ", "equity", "US", "USD", "Johnson & Johnson"),
    ("US_EQ_PG", "PG", "equity", "US", "USD", "Procter & Gamble"),
    ("US_EQ_COST", "COST", "equity", "US", "USD", "Costco"),
    ("US_EQ_KO", "KO", "equity", "US", "USD", "Coca-Cola"),

    # ETFs
    ("ETF_US_SPY", "SPY", "etf", "US", "USD", "S&P 500 ETF"),
    ("ETF_US_QQQ", "QQQ", "etf", "US", "USD", "Nasdaq 100 ETF"),
    ("ETF_US_IWM", "IWM", "etf", "US", "USD", "Russell 2000 ETF"),
    ("ETF_US_EEM", "EEM", "etf", "US", "USD", "Emerging Markets ETF"),
    ("ETF_US_GLD", "GLD", "etf", "US", "USD", "Gold ETF"),
    ("ETF_US_TLT", "TLT", "etf", "US", "USD", "20+ Year Treasury ETF"),
    ("ETF_US_IEF", "IEF", "etf", "US", "USD", "7-10 Year Treasury ETF"),
    ("ETF_US_HYG", "HYG", "etf", "US", "USD", "High Yield Bond ETF"),
    ("ETF_US_VNQ", "VNQ", "etf", "US", "USD", "US REIT ETF"),
    ("ETF_BR_BOVA11", "BOVA11.SA", "etf", "BR", "BRL", "Ibovespa ETF"),
    ("ETF_BR_IVVB11", "IVVB11.SA", "etf", "BR", "BRL", "S&P 500 ETF BR"),

    # Índices
    ("IDX_US_SP500", "^GSPC", "index", "US", "INDEX", "S&P 500"),
    ("IDX_US_NASDAQ", "^IXIC", "index", "US", "INDEX", "Nasdaq Composite"),
    ("IDX_BR_IBOV", "^BVSP", "index", "BR", "INDEX", "Ibovespa"),
    ("IDX_US_VIX", "^VIX", "index", "US", "INDEX", "CBOE VIX"),

    # Câmbio
    ("FX_USDBRL", "BRL=X", "fx", "GLOBAL", "FX", "USD/BRL"),
    ("FX_EURUSD", "EURUSD=X", "fx", "GLOBAL", "FX", "EUR/USD"),
    ("FX_USDJPY", "JPY=X", "fx", "GLOBAL", "FX", "USD/JPY"),

    # Cripto
    ("CRYPTO_BTCUSD", "BTC-USD", "crypto", "GLOBAL", "USD", "Bitcoin"),
    ("CRYPTO_ETHUSD", "ETH-USD", "crypto", "GLOBAL", "USD", "Ethereum"),

    # Commodities
    ("CMDTY_GOLD", "GC=F", "commodity", "GLOBAL", "USD", "Gold Futures"),
    ("CMDTY_SILVER", "SI=F", "commodity", "GLOBAL", "USD", "Silver Futures"),
    ("CMDTY_OIL_WTI", "CL=F", "commodity", "GLOBAL", "USD", "WTI Crude Oil Futures"),
]

asset_registry = pd.DataFrame(
    ASSETS,
    columns=["asset_id", "yahoo_ticker", "asset_class", "region", "quote_currency", "display_name"],
)
asset_registry["provider"] = PROVIDER
asset_registry["dataset_version"] = DATASET_VERSION
asset_registry["active"] = True

assert asset_registry["asset_id"].is_unique
assert asset_registry["yahoo_ticker"].is_unique

registry_file = DIRS["registry"] / f"asset_registry__{DATASET_VERSION}.csv"
asset_registry.to_csv(registry_file, index=False)

display(asset_registry)
print("Ativos cadastrados:", len(asset_registry))
print("Salvo em:", registry_file)


In [ ]:
# CÉLULA 4 — seleção do universo sem alterar os nomes canônicos

def select_universe(registry: pd.DataFrame, mode: str) -> pd.DataFrame:
    r = registry.copy()
    if mode == "MIXED_GLOBAL":
        keep = r["asset_class"].isin(["equity", "etf", "index", "fx", "crypto", "commodity"])
    elif mode == "EQUITY_ONLY":
        keep = r["asset_class"].eq("equity")
    elif mode == "BRAZIL_ONLY":
        keep = r["region"].eq("BR")
    elif mode == "US_LIQUID":
        keep = r["region"].eq("US") & r["asset_class"].isin(["equity", "etf", "index"])
    elif mode == "MULTI_ASSET":
        keep = r["asset_class"].isin(["etf", "index", "fx", "crypto", "commodity"])
    else:
        raise ValueError(f"UNIVERSE_MODE desconhecido: {mode}")
    return r.loc[keep].reset_index(drop=True)

universe_registry = select_universe(asset_registry, UNIVERSE_MODE)
print("UNIVERSE_MODE:", UNIVERSE_MODE)
print("N ativos candidatos:", len(universe_registry))
display(universe_registry[["asset_id", "yahoo_ticker", "asset_class", "region"]])


## Célula 5 — downloader robusto do Yahoo Finance

Cada ticker é baixado separadamente. Isso evita que uma falha em um ativo invalide todo o lote.


In [ ]:
# CÉLULA 5 — DOWNLOAD ROBUSTO

def safe_download_one(ticker: str, start: str, end: str, retries: int = 3) -> pd.DataFrame:
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            df = yf.download(
                ticker,
                start=start,
                end=end,
                auto_adjust=False,
                progress=False,
                actions=False,
                threads=False,
            )
            if df is None or df.empty:
                raise RuntimeError("download vazio")
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
            df = df.copy()
            df.index = pd.to_datetime(df.index).tz_localize(None)
            return df
        except Exception as exc:
            last_err = exc
            time.sleep(1.5 * attempt)
    raise RuntimeError(f"Falha em {ticker}: {last_err}")

DEV_START = f"{TRAIN_YEAR}-01-01"
DEV_END = f"{TRAIN_YEAR + 1}-01-01"
TEST_START = f"{TEST_YEAR}-01-01"
TEST_END = f"{TEST_YEAR + 1}-01-01"
FULL_START = min(DEV_START, TEST_START)
FULL_END = max(DEV_END, TEST_END)

raw_frames = {}
download_log = []

for row in universe_registry.itertuples(index=False):
    try:
        df = safe_download_one(row.yahoo_ticker, FULL_START, FULL_END)
        df["asset_id"] = row.asset_id
        df["yahoo_ticker"] = row.yahoo_ticker
        raw_frames[row.asset_id] = df
        out = DIRS["raw"] / f"yf__raw__{row.asset_id}__{DATASET_VERSION}.parquet"
        df.to_parquet(out)
        download_log.append({
            "asset_id": row.asset_id,
            "yahoo_ticker": row.yahoo_ticker,
            "status": "ok",
            "n_rows": len(df),
            "first_date": str(df.index.min().date()),
            "last_date": str(df.index.max().date()),
            "error": "",
        })
    except Exception as exc:
        download_log.append({
            "asset_id": row.asset_id,
            "yahoo_ticker": row.yahoo_ticker,
            "status": "failed",
            "n_rows": 0,
            "first_date": "",
            "last_date": "",
            "error": str(exc),
        })

download_log_df = pd.DataFrame(download_log)
download_log_df.to_csv(DIRS["manifests"] / f"download_log__{DATASET_VERSION}.csv", index=False)
display(download_log_df)
print(download_log_df["status"].value_counts(dropna=False))


In [ ]:
# CÉLULA 6 — matriz de preços canônica

def select_price_column(df: pd.DataFrame) -> pd.Series:
    if "Adj Close" in df.columns and df["Adj Close"].notna().any():
        return df["Adj Close"].astype(float)
    if "Close" in df.columns:
        return df["Close"].astype(float)
    raise KeyError("Nem 'Adj Close' nem 'Close' disponíveis.")

series = {}
for asset_id, df in raw_frames.items():
    try:
        series[asset_id] = select_price_column(df).rename(asset_id)
    except Exception as exc:
        print("Ignorado:", asset_id, exc)

prices_all = pd.concat(series.values(), axis=1).sort_index()
assert set(prices_all.columns).issubset(set(asset_registry["asset_id"]))

prices_all.to_parquet(DIRS["processed"] / f"yf__prices__all__{DATASET_VERSION}.parquet")
if SAVE_CSV_TOO:
    prices_all.to_csv(DIRS["processed"] / f"yf__prices__all__{DATASET_VERSION}.csv")

print("shape:", prices_all.shape)
display(prices_all.tail())


## Célula 7 — split temporal sem vazamento e calendários heterogêneos

A elegibilidade do ativo é decidida **apenas com dados do ano de desenvolvimento**.

**Correção v2:** ações, índices, FX, commodities e cripto não compartilham o mesmo calendário.
Portanto, a elegibilidade não é mais calculada como fração de `prices_all.notna()` sobre o
índice global. Cada ativo é auditado em seu **calendário observado**, e os retornos são
calculados sobre observações consecutivas do próprio ativo antes do alinhamento entre ativos.

Isso evita dois erros:
1. penalizar ações por fins de semana/feriados de cripto;
2. criar retornos artificiais zero por `ffill()` em dias sem negociação.


In [ ]:
# CÉLULA 7 — SPLITS TEMPORAIS + CALENDÁRIOS HETEROGÊNEOS

train_start = pd.Timestamp(f"{TRAIN_YEAR}-01-01")
cal_start   = pd.Timestamp(f"{TRAIN_YEAR}-{CALIBRATION_START_MONTH:02d}-01")
dev_end     = pd.Timestamp(f"{TRAIN_YEAR + 1}-01-01")
test_start  = pd.Timestamp(f"{TEST_YEAR}-01-01")
test_end    = pd.Timestamp(f"{TEST_YEAR + 1}-01-01")

prices_train_raw = prices_all.loc[
    (prices_all.index >= train_start) & (prices_all.index < cal_start)
].copy()

prices_cal_raw = prices_all.loc[
    (prices_all.index >= cal_start) & (prices_all.index < dev_end)
].copy()

prices_test_raw = prices_all.loc[
    (prices_all.index >= test_start) & (prices_all.index < test_end)
].copy()

prices_dev_raw = prices_all.loc[
    (prices_all.index >= train_start) & (prices_all.index < dev_end)
].copy()


def _observed_span_stats(series: pd.Series):
    s = series.dropna()
    if s.empty:
        return {
            "n_obs": 0,
            "first_date": pd.NaT,
            "last_date": pd.NaT,
            "span_days": 0,
            "calendar_density_diagnostic": 0.0,
        }

    first_date = s.index.min()
    last_date = s.index.max()
    span_days = int((last_date - first_date).days + 1)

    return {
        "n_obs": int(len(s)),
        "first_date": first_date,
        "last_date": last_date,
        # Apenas diagnóstico. NÃO é critério de elegibilidade,
        # pois calendários naturais diferem entre classes de ativos.
        "span_days": span_days,
        "calendar_density_diagnostic": float(len(s) / max(span_days, 1)),
    }


eligibility_rows = []

for asset_id in prices_dev_raw.columns:
    dev_stats = _observed_span_stats(prices_dev_raw[asset_id])
    n_train = int(prices_train_raw[asset_id].notna().sum())
    n_cal = int(prices_cal_raw[asset_id].notna().sum())

    if dev_stats["n_obs"] == 0:
        starts_near_beginning = False
        ends_near_end = False
    else:
        starts_near_beginning = bool(
            dev_stats["first_date"] <= train_start + pd.Timedelta(days=BOUNDARY_TOLERANCE_DAYS)
        )
        ends_near_end = bool(
            dev_stats["last_date"] >= dev_end - pd.Timedelta(days=BOUNDARY_TOLERANCE_DAYS)
        )

    enough_dev = dev_stats["n_obs"] >= MIN_DEV_OBSERVATIONS
    enough_train = n_train >= MIN_TRAIN_OBSERVATIONS
    enough_cal = n_cal >= MIN_CALIBRATION_OBSERVATIONS

    eligible = bool(
        starts_near_beginning
        and ends_near_end
        and enough_dev
        and enough_train
        and enough_cal
    )

    reasons = []
    if dev_stats["n_obs"] == 0:
        reasons.append("no_development_data")
    if not starts_near_beginning:
        reasons.append("late_start")
    if not ends_near_end:
        reasons.append("early_end")
    if not enough_dev:
        reasons.append("too_few_dev_observations")
    if not enough_train:
        reasons.append("too_few_train_observations")
    if not enough_cal:
        reasons.append("too_few_calibration_observations")
    if eligible:
        reasons = ["ok"]

    eligibility_rows.append({
        "asset_id": asset_id,
        "n_dev_observations": dev_stats["n_obs"],
        "n_train_observations": n_train,
        "n_calibration_observations": n_cal,
        "first_development_date": dev_stats["first_date"],
        "last_development_date": dev_stats["last_date"],
        "development_span_days": dev_stats["span_days"],
        "calendar_density_diagnostic": dev_stats["calendar_density_diagnostic"],
        "starts_near_development_beginning": starts_near_beginning,
        "ends_near_development_end": ends_near_end,
        "eligible": eligible,
        "reason": "|".join(reasons),
    })

eligibility_df = pd.DataFrame(eligibility_rows).sort_values(
    ["eligible", "n_dev_observations", "asset_id"],
    ascending=[False, False, True],
).reset_index(drop=True)

eligible_assets = eligibility_df.loc[
    eligibility_df["eligible"], "asset_id"
].tolist()

if len(eligible_assets) < min(N_VALUES):
    failed_preview = eligibility_df.loc[
        ~eligibility_df["eligible"],
        ["asset_id", "n_dev_observations", "n_train_observations",
         "n_calibration_observations", "reason"],
    ].head(20)

    print("Ativos elegíveis:", len(eligible_assets), "/", len(eligibility_df))
    display(failed_preview)

    raise RuntimeError(
        f"Apenas {len(eligible_assets)} ativos passaram a elegibilidade por calendário natural, "
        f"mas N_VALUES exige pelo menos {min(N_VALUES)}. "
        "Revise o download_log e asset_eligibility antes de reduzir os limiares."
    )


# ---------------------------------------------------------------------
# Preços: preservar NaN real em dias em que o ativo não negociou.
# NÃO aplicar ffill no calendário global.
# ---------------------------------------------------------------------

prices_train = prices_train_raw[eligible_assets].copy()
prices_cal   = prices_cal_raw[eligible_assets].copy()
prices_test  = prices_test_raw[eligible_assets].copy()


# ---------------------------------------------------------------------
# Retornos: calcular em observações consecutivas DO PRÓPRIO ATIVO.
# Ex.: segunda-feira de uma ação usa a sexta-feira anterior, mesmo que
# sábado/domingo existam no índice global por causa de cripto.
# ---------------------------------------------------------------------

def natural_calendar_returns(price_frame: pd.DataFrame) -> pd.DataFrame:
    return_series = []

    for asset_id in price_frame.columns:
        s = price_frame[asset_id].dropna().astype(float)

        if len(s) < 2:
            continue

        r = (
            s.pct_change(fill_method=None)
            .replace([np.inf, -np.inf], np.nan)
            .rename(asset_id)
        )
        return_series.append(r)

    if not return_series:
        return pd.DataFrame(index=price_frame.index)

    return pd.concat(return_series, axis=1).sort_index()


returns_train = natural_calendar_returns(prices_train)
returns_cal   = natural_calendar_returns(prices_cal)
returns_test  = natural_calendar_returns(prices_test)


split_map = {
    "train": (prices_train, returns_train),
    "calibration": (prices_cal, returns_cal),
    "test": (prices_test, returns_test),
}

for split_name, (p, r) in split_map.items():
    p.to_parquet(
        DIRS["splits"] / f"yf__prices__{split_name}__{DATASET_VERSION}.parquet"
    )
    r.to_parquet(
        DIRS["splits"] / f"yf__returns__{split_name}__{DATASET_VERSION}.parquet"
    )

    if SAVE_CSV_TOO:
        p.to_csv(
            DIRS["splits"] / f"yf__prices__{split_name}__{DATASET_VERSION}.csv"
        )
        r.to_csv(
            DIRS["splits"] / f"yf__returns__{split_name}__{DATASET_VERSION}.csv"
        )


eligibility_path = (
    DIRS["registry"] / f"asset_eligibility__{DATASET_VERSION}.csv"
)
eligibility_df.to_csv(eligibility_path, index=False)


# Auditoria adicional: número de retornos efetivamente disponíveis por ativo.
return_availability_df = pd.DataFrame({
    "asset_id": eligible_assets,
    "n_returns_train": [
        int(returns_train[a].notna().sum()) for a in eligible_assets
    ],
    "n_returns_calibration": [
        int(returns_cal[a].notna().sum()) for a in eligible_assets
    ],
    "n_returns_test": [
        int(returns_test[a].notna().sum()) for a in eligible_assets
    ],
})
return_availability_df.to_csv(
    DIRS["registry"] / f"return_availability__{DATASET_VERSION}.csv",
    index=False,
)


print("Ativos elegíveis:", len(eligible_assets), "/", len(eligibility_df))
print("train:", prices_train.index.min(), "->", prices_train.index.max(), prices_train.shape)
print("calibration:", prices_cal.index.min(), "->", prices_cal.index.max(), prices_cal.shape)
print("test:", prices_test.index.min(), "->", prices_test.index.max(), prices_test.shape)
print("Elegibilidade salva em:", eligibility_path)

display(eligibility_df.head(50))
display(return_availability_df.head(50))


## Célula 8 — IDs estáveis

- `parent_instance_id`: define o conjunto físico de ativos e a cardinalidade.
- `scenario_uid`: define uma perturbação específica desse parent.


In [ ]:
# CÉLULA 8 — FUNÇÕES DE NOMENCLATURA E HASH

def short_hash(text: str, n: int = 16) -> str:
    return sha256(text.encode("utf-8")).hexdigest()[:n]

def canonical_parent_name(train_year: int, test_year: int, n: int, k: int, rep: int) -> str:
    return f"YF_{train_year}_{test_year}__N{n:02d}__K{k:02d}__P{rep:04d}"

def parent_instance_id(asset_ids, train_year: int, test_year: int, n: int, k: int, rep: int) -> str:
    ordered = "|".join(sorted(asset_ids))
    raw = f"{DATASET_VERSION}|{train_year}|{test_year}|{n}|{k}|{rep}|{ordered}"
    return short_hash(raw)

def scenario_uid(parent_id: str, split: str, shock_asset_id: str, delta: float) -> str:
    raw = f"{parent_id}|{split}|{shock_asset_id}|{delta:+.8f}"
    return short_hash(raw)

print(canonical_parent_name(TRAIN_YEAR, TEST_YEAR, 10, 5, 0))


## Célula 9 — alocação reprodutível de `n` e `k`

A alocação é definida uma única vez e salva em manifesto. O ano de teste **não altera os parent instances**.


In [ ]:
# CÉLULA 9 — PARENT INSTANCES

rng = np.random.default_rng(RANDOM_SEED)
eligible_registry = asset_registry[asset_registry["asset_id"].isin(eligible_assets)].copy().reset_index(drop=True)

parent_rows = []
for n in N_VALUES:
    if n > len(eligible_registry):
        print(f"[skip] n={n}: apenas {len(eligible_registry)} ativos elegíveis")
        continue
    for k_over_n in K_OVER_N_VALUES:
        k = int(round(n * k_over_n))
        k = max(1, min(k, n - 1))
        for rep in range(N_PARENT_PER_COMBO):
            chosen_idx = rng.choice(len(eligible_registry), size=n, replace=False)
            chosen = eligible_registry.iloc[np.sort(chosen_idx)].copy()
            asset_ids = chosen["asset_id"].tolist()
            yahoo_tickers = chosen["yahoo_ticker"].tolist()
            classes = chosen["asset_class"].tolist()
            regions = chosen["region"].tolist()
            pname = canonical_parent_name(TRAIN_YEAR, TEST_YEAR, n, k, rep)
            pid = parent_instance_id(asset_ids, TRAIN_YEAR, TEST_YEAR, n, k, rep)
            parent_rows.append({
                "parent_instance_id": pid,
                "parent_name": pname,
                "dataset_version": DATASET_VERSION,
                "provider": PROVIDER,
                "train_year": TRAIN_YEAR,
                "calibration_start_month": CALIBRATION_START_MONTH,
                "test_year": TEST_YEAR,
                "universe_mode": UNIVERSE_MODE,
                "n_assets": n,
                "k": k,
                "k_over_n": k / n,
                "parent_rep": rep,
                "asset_ids": "|".join(asset_ids),
                "yahoo_tickers": "|".join(yahoo_tickers),
                "asset_classes": "|".join(classes),
                "regions": "|".join(regions),
                "allocation_seed": RANDOM_SEED,
            })

parent_instances = pd.DataFrame(parent_rows)
assert parent_instances["parent_instance_id"].is_unique
assert parent_instances["parent_name"].is_unique

parent_file = DIRS["instances"] / f"parent_instances__{DATASET_VERSION}.csv"
parent_instances.to_csv(parent_file, index=False)
print("Parent instances:", len(parent_instances))
display(parent_instances.head(20))
print("Salvo em:", parent_file)


## Célula 10 — estatísticas \(\mu\) e \(\Sigma\) por parent instance

A estimação é feita **somente no treino**.

Na v2, cada parent registra também `n_joint_observations`: o número de datas em que
**todos os ativos daquele Hamiltoniano** possuem retorno observado. Isso separa claramente:

- elegibilidade individual do ativo em seu calendário natural;
- amostra conjunta realmente usada para estimar \(\mu\) e \(\Sigma\).


In [ ]:
# CÉLULA 10 — FEATURES DE TREINO POR PARENT + AUDITORIA N_joint

feature_rows = []
cov_rows = []
parent_training_audit_rows = []

for row in parent_instances.itertuples(index=False):
    asset_ids = row.asset_ids.split("|")

    # Interseção somente no momento em que o Hamiltoniano exige uma
    # matriz de covariância conjunta.
    r = returns_train[asset_ids].dropna(how="any")
    n_joint_observations = int(len(r))

    if n_joint_observations < MIN_JOINT_TRAIN_OBSERVATIONS:
        parent_training_audit_rows.append({
            "parent_instance_id": row.parent_instance_id,
            "parent_name": row.parent_name,
            "n_assets": row.n_assets,
            "k": row.k,
            "k_over_n": row.k_over_n,
            "n_joint_observations": n_joint_observations,
            "min_required": MIN_JOINT_TRAIN_OBSERVATIONS,
            "status": "skipped_insufficient_joint_observations",
        })
        continue

    mu = r.mean()
    sigma = r.cov()

    parent_training_audit_rows.append({
        "parent_instance_id": row.parent_instance_id,
        "parent_name": row.parent_name,
        "n_assets": row.n_assets,
        "k": row.k,
        "k_over_n": row.k_over_n,
        "n_joint_observations": n_joint_observations,
        "min_required": MIN_JOINT_TRAIN_OBSERVATIONS,
        "status": "ok",
    })

    for asset_id in asset_ids:
        feature_rows.append({
            "parent_instance_id": row.parent_instance_id,
            "parent_name": row.parent_name,
            "asset_id": asset_id,
            "n_assets": row.n_assets,
            "k": row.k,
            "k_over_n": row.k_over_n,
            "n_joint_observations": n_joint_observations,
            "mu_daily": float(mu.loc[asset_id]),
            "variance_daily": float(sigma.loc[asset_id, asset_id]),
        })

    for a_i in asset_ids:
        for a_j in asset_ids:
            cov_rows.append({
                "parent_instance_id": row.parent_instance_id,
                "parent_name": row.parent_name,
                "n_assets": row.n_assets,
                "k": row.k,
                "n_joint_observations": n_joint_observations,
                "asset_i": a_i,
                "asset_j": a_j,
                "covariance_daily": float(sigma.loc[a_i, a_j]),
            })


parent_asset_features = pd.DataFrame(feature_rows)
parent_covariances = pd.DataFrame(cov_rows)
parent_training_audit = pd.DataFrame(parent_training_audit_rows)

if parent_asset_features.empty:
    raise RuntimeError(
        "Nenhum parent instance possui observações conjuntas suficientes para estimar mu/Sigma. "
        "Audite parent_training_audit antes de alterar MIN_JOINT_TRAIN_OBSERVATIONS."
    )

parent_asset_features.to_parquet(
    DIRS["instances"] / f"parent_asset_features__train__{DATASET_VERSION}.parquet",
    index=False,
)
parent_covariances.to_parquet(
    DIRS["instances"] / f"parent_covariances__train__{DATASET_VERSION}.parquet",
    index=False,
)
parent_training_audit.to_csv(
    DIRS["manifests"] / f"parent_training_audit__{DATASET_VERSION}.csv",
    index=False,
)

if SAVE_CSV_TOO:
    parent_asset_features.to_csv(
        DIRS["instances"] / f"parent_asset_features__train__{DATASET_VERSION}.csv",
        index=False,
    )
    parent_covariances.to_csv(
        DIRS["instances"] / f"parent_covariances__train__{DATASET_VERSION}.csv",
        index=False,
    )

print("asset features:", parent_asset_features.shape)
print("covariance rows:", parent_covariances.shape)
print("parents auditados:", parent_training_audit.shape)
print(parent_training_audit["status"].value_counts(dropna=False))

display(parent_asset_features.head())
display(
    parent_training_audit.sort_values(
        ["status", "n_joint_observations"],
        ascending=[True, False],
    ).head(50)
)


## Célula 11 — manifesto de splits do Transformer / Conformal

Além do split temporal, definimos:

```text
in_distribution: n <= 16
ood_size:        n in {18, 20}
```


In [ ]:
# CÉLULA 11 — SPLIT MANIFEST

split_manifest = pd.DataFrame([
    {"split": "train", "start": str(train_start.date()), "end_exclusive": str(cal_start.date()), "role": "model_fit", "may_fit_model": True, "may_calibrate_conformal": False, "may_evaluate_final": False},
    {"split": "calibration", "start": str(cal_start.date()), "end_exclusive": str(dev_end.date()), "role": "conformal_calibration", "may_fit_model": False, "may_calibrate_conformal": True, "may_evaluate_final": False},
    {"split": "test", "start": str(test_start.date()), "end_exclusive": str(test_end.date()), "role": "temporal_holdout", "may_fit_model": False, "may_calibrate_conformal": False, "may_evaluate_final": True},
])
split_manifest.to_csv(DIRS["manifests"] / f"split_manifest__{DATASET_VERSION}.csv", index=False)

size_generalization = parent_instances[["parent_instance_id", "parent_name", "n_assets", "k", "k_over_n"]].copy()
size_generalization["size_regime"] = np.where(size_generalization["n_assets"] <= 16, "in_distribution", "ood_size")
size_generalization.to_csv(DIRS["manifests"] / f"size_generalization_manifest__{DATASET_VERSION}.csv", index=False)

display(split_manifest)
display(size_generalization["size_regime"].value_counts())


## Célula 12 — função única para criar cenários futuros

Esta função deve ser reutilizada por 20.22B/20.23 para manter a nomenclatura compatível.


In [ ]:
# CÉLULA 12 — CRIADOR CANÔNICO DE CENÁRIOS

def build_scenario_record(parent_row: pd.Series, split: str, shock_asset_id: str, delta_from_boundary: float, scenario_family: str):
    asset_ids = str(parent_row["asset_ids"]).split("|")
    if shock_asset_id not in asset_ids:
        raise ValueError(f"{shock_asset_id} não pertence ao parent {parent_row['parent_instance_id']}")

    uid = scenario_uid(str(parent_row["parent_instance_id"]), split, shock_asset_id, float(delta_from_boundary))
    return {
        "scenario_uid": uid,
        "parent_instance_id": str(parent_row["parent_instance_id"]),
        "parent_name": str(parent_row["parent_name"]),
        "scenario_family": scenario_family,
        "split": split,
        "shock_asset_id": shock_asset_id,
        "delta_from_boundary": float(delta_from_boundary),
        "n_assets": int(parent_row["n_assets"]),
        "k": int(parent_row["k"]),
        "k_over_n": float(parent_row["k_over_n"]),
        "dataset_version": DATASET_VERSION,
        "provider": PROVIDER,
    }

example_parent = parent_instances.iloc[0]
example_asset = example_parent["asset_ids"].split("|")[0]
example_scenario = build_scenario_record(example_parent, "train", example_asset, -0.05, "near_return_boundary")
example_scenario


## Célula 13 — manifesto global

In [ ]:
# CÉLULA 13 — DATASET MANIFEST

manifest = {
    "dataset_version": DATASET_VERSION,
    "provider": PROVIDER,
    "created_at_local": datetime.now().isoformat(timespec="seconds"),
    "train_year": TRAIN_YEAR,
    "test_year": TEST_YEAR,
    "calibration_start_month": CALIBRATION_START_MONTH,
    "universe_mode": UNIVERSE_MODE,
    "eligibility_policy": {
        "calendar": "natural_per_asset",
        "min_dev_observations": MIN_DEV_OBSERVATIONS,
        "min_train_observations": MIN_TRAIN_OBSERVATIONS,
        "min_calibration_observations": MIN_CALIBRATION_OBSERVATIONS,
        "boundary_tolerance_days": BOUNDARY_TOLERANCE_DAYS,
        "global_calendar_coverage_used": False,
        "global_ffill_used": False,
    },
    "joint_estimation_policy": {
        "min_joint_train_observations": MIN_JOINT_TRAIN_OBSERVATIONS,
        "joint_dates_defined_per_parent": True,
    },
    "n_values": N_VALUES,
    "k_over_n_values": K_OVER_N_VALUES,
    "n_parent_per_combo": N_PARENT_PER_COMBO,
    "random_seed": RANDOM_SEED,
    "n_registry_assets": int(len(asset_registry)),
    "n_universe_assets": int(len(universe_registry)),
    "n_eligible_assets": int(len(eligible_assets)),
    "n_parent_instances": int(len(parent_instances)),
    "n_parent_instances_with_training_features": int(
        parent_training_audit["status"].eq("ok").sum()
    ),
    "naming": {
        "asset_id": "<REGION_OR_CLASS>_<TYPE>_<SYMBOL>",
        "parent_name": "YF_<trainyear>_<testyear>__N<nn>__K<kk>__P<rrrr>",
        "parent_instance_id": "sha256[:16]",
        "scenario_uid": "sha256(parent|split|shock_asset|delta)[:16]",
        "file_prefix": f"yf__<content>__<split>__{DATASET_VERSION}",
    },
    "anti_leakage_rules": [
        "test year is never used for asset eligibility",
        "test year is never used to fit mu/Sigma",
        "calibration split is not used to fit model weights",
        "parent_instance_id groups all derived scenarios",
        "canonical asset_id is independent from Yahoo ticker",
        "asset eligibility uses each asset natural observed calendar",
        "no global forward-fill is used to manufacture closed-market observations",
        "returns are computed on consecutive observed prices per asset before cross-asset alignment",
    ],
}

manifest_path = DIRS["manifests"] / f"dataset_manifest__{DATASET_VERSION}.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(json.dumps(manifest, indent=2, ensure_ascii=False))
print("Manifest:", manifest_path)


# Validações obrigatórias

In [ ]:
# CÉLULA 14 — ASSERTS DE AUDITORIA

assert asset_registry["asset_id"].is_unique
assert asset_registry["yahoo_ticker"].is_unique
assert parent_instances["parent_instance_id"].is_unique
assert parent_instances["parent_name"].is_unique

for r in parent_instances.itertuples(index=False):
    assert len(r.asset_ids.split("|")) == r.n_assets
    assert 0 < r.k < r.n_assets

assert test_start >= dev_end
assert set(prices_all.columns).issubset(set(asset_registry["asset_id"]))
assert set(eligible_assets).issubset(set(prices_all.columns))
assert bool(eligibility_df.loc[eligibility_df["eligible"], "eligible"].all())

# Anti-vazamento temporal
assert prices_train.index.max() < test_start
assert prices_cal.index.max() < test_start

# A v2 não usa ffill global. Em um calendário misto, NaNs legítimos devem existir
# se houver classes com calendários distintos.
assert returns_train.columns.is_unique
assert returns_cal.columns.is_unique
assert returns_test.columns.is_unique

# Parents que chegaram a mu/Sigma respeitam o limiar conjunto.
ok_parent_audit = parent_training_audit[
    parent_training_audit["status"].eq("ok")
]
assert (ok_parent_audit["n_joint_observations"] >= MIN_JOINT_TRAIN_OBSERVATIONS).all()

print("AUDITORIA ESTRUTURAL v2: PASS")
print("Ativos elegíveis:", len(eligible_assets))
print("Parents com mu/Sigma:", len(ok_parent_audit), "/", len(parent_instances))


# Próximos notebooks

### 20.22B — fronteiras e seleção ativa
Calcular fronteiras clássicas e selecionar casos próximos de \(|\delta|\lesssim 0.1\).

### 20.22C — campanha VQE/QGT/cold-start
Calcular apenas nos casos informativos: \(P_{\rm opt}\), QGT rank, participação, parâmetros ativos, rota dominante, multiplicidade de bacias e cold-start success.

### 20.23 — Transformer + Conformal Prediction

Entradas permitidas: \(\mu_i\), \(\Sigma_{ii}\), conexões \(\Sigma_{ij}\), \(k/n\) e metadados estruturais conhecidos antes da solução.

Targets: bitstring ótimo, energia, QGT, dificuldade variacional e regime/bacia.

Conformal: intervalo de energia, conjunto conformal de bitstrings, Mondrian por \(n\), \(k/n\), classe de ativo e regime, além de OOD em \(n=18,20\).
